# Webinar-Ready: SQL Joins with SQLite Magic (90 minutes) — insurance.db

**Goal:** teach SQL joins (basic → advanced) using a real dataset, with hands-on practice.  
**SQL runs inside the notebook** using `%%sql` magic (SQLite).

## What learners need (before we start)
1. VS Code extensions: **Jupyter**
2. Python packages:
   ```bash
   pip install ipython-sql sqlalchemy
   ```
3. Files in the same folder as this notebook:
   - `insurance.db`  *(created by `convert_insurance_to_db.py`)*

---

## Webinar Flow

| Time | Segment | Output |
|---:|---|---|
| 0–5 | Setup & connect to DB | Everyone can run `%%sql` |
| 5–15 | Assessment (attempt only) | Learners try Q1–Q4 |
| 15–25 | Theory recap + join intuition | Clear mental model |
| 25–40 | Basic joins: INNER + LEFT | Read outputs correctly |
| 40–55 | Intermediate: RIGHT/FULL OUTER (SQLite emulation) | Demo NULLs/orphans |
| 55–70 | Advanced: CROSS JOIN + SELF JOIN | Grids + pair matching |
| 70–85 | Advanced: window functions patterns | Ranking / percentiles |
| 85–90 | Reveal solutions + wrap-up | Debrief |


#  SQL Joins Theory 



## Why joins exist
- Databases split data into multiple tables to reduce duplication and keep it consistent.
- A **JOIN** combines rows from two tables using a condition (usually matching keys).

## Keys and grain (most important concept)
- **Primary key (PK):** uniquely identifies a row.
- **Foreign key (FK):** points to a row in another table.
- **Grain:** what a row represents (one person? one claim? one payment?).  
If grain is unclear, joins often create duplicates.

## INNER JOIN
- Returns **only rows that match** on both sides.

## LEFT JOIN
- Returns **all rows from the left** table.
- Non-matches on the right become **NULL**.

## RIGHT JOIN (SQLite note)
- SQLite doesn’t support RIGHT JOIN.
- Emulate by swapping tables and using LEFT JOIN.

## FULL OUTER JOIN (SQLite note)
- SQLite doesn’t support FULL OUTER JOIN.
- Emulate using `UNION ALL` of two LEFT JOINs (plus a filter for the unmatched side).

## CROSS JOIN
- All combinations (Cartesian product).
- Great for reporting grids (e.g., region × smoker).

## SELF JOIN
- A table joined to itself.
- Useful for comparisons (similarity, duplicates, hierarchies).

## Pitfalls
- Wrong key ⇒ duplicates (accidental many-to-many).
- NULL never matches `=`.
- Always sanity-check row counts.


## 0) Setup (0–5 min)
Run these two cells first.

In [ ]:
%load_ext sql
%sql sqlite:///insurance.db


In [ ]:
%%sql
-- Quick check: list tables and views
SELECT name, type
FROM sqlite_master
WHERE type IN ('table','view')
ORDER BY type, name;


# Assessment (Attempt First) — 5–15 min

**Rules for this section:**
- Try each question **without** looking at the solution.
- After time is up, we’ll reveal solutions together.

> Tip: If you get stuck, start from `vw_insurance_flat`.


## Q1 (Beginner): Average charges by smoker status
Return:
- smoker
- number of people
- average charges

**Expected:** smokers have higher average charges.


In [ ]:
%%sql
-- Q1: Average charges by smoker status
-- We aggregate (summarise) people by whether they smoke.

SELECT
  smoker,                          -- column we are grouping by (e.g., 'yes'/'no')
  COUNT(*) AS n_people,            -- COUNT(*) counts all rows in each smoker group
  ROUND(AVG(charges), 2) AS avg_charges  -- AVG computes mean; ROUND(...,2) keeps 2 decimals
FROM vw_insurance_flat             -- a flattened view that already joins person + dimensions + fact
GROUP BY smoker                    -- one output row per smoker value
ORDER BY avg_charges DESC;         -- sort so the highest average charges appear first


<details>
<summary><b>Solution (click to expand)</b></summary>

```sql
SELECT smoker,
       COUNT(*) AS n_people,
       ROUND(AVG(charges), 2) AS avg_charges
FROM vw_insurance_flat
GROUP BY smoker
ORDER BY avg_charges DESC;
```

### Explanation (read-out friendly)

**What this query answers:** “For smokers vs non-smokers, how many people are there, and what is the average medical charge?”

- `SELECT smoker, ...`  
  Picks the grouping column (`smoker`) and the summary calculations we want to show per group.

- `COUNT(*) AS n_people`  
  Counts **rows** in each group. `*` means “count every row”, not a specific column.  
  `AS n_people` gives the result column a friendly name.

- `AVG(charges)`  
  Calculates the mean of `charges` inside each group.

- `ROUND(AVG(charges), 2) AS avg_charges`  
  `ROUND(x, 2)` formats the numeric result to **2 decimal places** for readability.

- `FROM vw_insurance_flat`  
  Reads from the flat view (already joined/cleaned for analysis).

- `GROUP BY smoker`  
  This is the key step: it tells SQL to **aggregate** rows into one row per `smoker` value.  
  Rule of thumb: every column in `SELECT` must be either:
  1) inside an aggregate function (`COUNT`, `AVG`, etc.), or  
  2) listed in `GROUP BY`.

</details>


## Q2 (Intermediate): Average charges by region × smoker
Return:
- region
- smoker
- number of people
- average charges

**Expected:** 8 rows (4 regions × 2 smoker statuses).


In [ ]:
%%sql
-- Q2: Average charges by region × smoker
-- This is a “group by two dimensions” problem: region AND smoker.

SELECT
  region,                          -- 1st grouping column
  smoker,                          -- 2nd grouping column
  COUNT(*) AS n_people,            -- number of people in each (region, smoker) combination
  ROUND(AVG(charges), 2) AS avg_charges
FROM vw_insurance_flat
GROUP BY region, smoker            -- group by BOTH columns (the order here doesn't change results)
ORDER BY region, smoker;           -- tidy output: grouped rows appear together


<details>
<summary><b>Solution (click to expand)</b></summary>

```sql
SELECT region,
       smoker,
       COUNT(*) AS n_people,
       ROUND(AVG(charges), 2) AS avg_charges
FROM vw_insurance_flat
GROUP BY region, smoker
ORDER BY region, smoker;
```

### Explanation (read-out friendly)

**What this query answers:** “Within each region, how do smoker and non-smoker charges compare?”

- `SELECT region, smoker, ...`  
  We want a breakdown by **two dimensions**: `region` *and* `smoker`.

- `COUNT(*) AS n_people`  
  Counts how many people fall into each **(region, smoker)** combination.

- `ROUND(AVG(charges), 2) AS avg_charges`  
  Computes the average `charges` for each combination, rounded to 2 decimals.

- `FROM vw_insurance_flat`  
  Uses the flat analysis view so we can group without doing joins in this step.

- `GROUP BY region, smoker`  
  Creates one output row per unique pair: for example `('southeast', 'yes')`.  
  Adding more columns to `GROUP BY` gives you a *more granular* breakdown.

- `ORDER BY region, smoker`  
  Sorts the final table so results are easy to scan: region first, then smoker status.

</details>


## Q3 (Intermediate → Advanced): Top 5 charges per region
Return the top 5 charges **within each region**, including:
- person_id, region, smoker, bmi, charges

**Hint:** window function `ROW_NUMBER()`.


In [ ]:
%%sql
-- Q3: Top 5 charges per region
-- We rank rows WITHIN each region using a window function, then filter to the top 5.

WITH ranked AS (
  SELECT
    person_id,
    region,
    smoker,
    bmi,
    charges,

    -- ROW_NUMBER() assigns 1,2,3,... inside each region (because of PARTITION BY region)
    -- ORDER BY charges DESC means the biggest charge gets rn = 1.
    ROW_NUMBER() OVER (
      PARTITION BY region
      ORDER BY charges DESC
    ) AS rn

  FROM vw_insurance_flat
)

SELECT *
FROM ranked
WHERE rn <= 5                      -- keep only the first 5 rows per region
ORDER BY region, rn;               -- show regions together, and in rank order inside each region


<details>
<summary><b>Solution (click to expand)</b></summary>

```sql
WITH ranked AS (
  SELECT
    person_id, region, smoker, bmi, charges,
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY charges DESC) AS rn
  FROM vw_insurance_flat
)
SELECT *
FROM ranked
WHERE rn <= 5
ORDER BY region, rn;
```

### Explanation (read-out friendly)

**What this query answers:** “For each region, who are the top 5 most expensive people (by charges)?”

This uses a **CTE** plus a **window function**.

- `WITH ranked AS (...)`  
  A **Common Table Expression** (CTE). Think: “create a temporary named result set” you can query afterwards.  
  It keeps the logic readable.

Inside the CTE:

- `ROW_NUMBER() OVER (PARTITION BY region ORDER BY charges DESC) AS rn`  
  This is the key line.
  - `ROW_NUMBER()` assigns 1, 2, 3… to rows.
  - `OVER (...)` tells SQL we are doing a **window calculation** (it does *not* collapse rows like `GROUP BY`).
  - `PARTITION BY region` resets numbering **for each region** (each region starts at 1).
  - `ORDER BY charges DESC` sorts within each region from highest charges to lowest, so `rn = 1` is the most expensive person in that region.

After the CTE:

- `SELECT * FROM ranked`  
  Pulls all columns from the ranked result, including `rn`.

- `WHERE rn <= 5`  
  Filters down to only the **top 5** per region.

- `ORDER BY region, rn`  
  Shows results region-by-region, with rank order inside each region.

</details>


## Q4 (Advanced): Top 10% charges within each region (deciles)
Mark the top 10% highest charges **per region** and summarise by region × smoker:
- n_people
- n_top_10pct
- avg_top_10pct_charges

**Hint:** `NTILE(10)`.


In [ ]:
%%sql
-- Q4: Top 10% charges within each region (deciles)
-- NTILE(10) splits each region into 10 equal-sized buckets by charges (approx equal row counts).
-- Decile 10 ≈ top 10% (highest charges) in that region.

WITH bucketed AS (
  SELECT
    region,
    smoker,
    charges,

    -- NTILE(10) labels rows 1..10 within each region after sorting by charges (ascending here).
    -- The highest charges end up in decile = 10.
    NTILE(10) OVER (
      PARTITION BY region
      ORDER BY charges
    ) AS decile

  FROM vw_insurance_flat
)

SELECT
  region,
  smoker,
  COUNT(*) AS n_people,            -- total people in the region+smoker group

  -- Count how many rows fall in the top decile (decile = 10).
  -- SUM over 1/0 is a common SQL counting trick.
  SUM(CASE WHEN decile = 10 THEN 1 ELSE 0 END) AS n_top_10pct,

  -- AVG ignores NULLs, so we only average charges for the top decile by returning charges
  -- when decile = 10, and NULL otherwise.
  ROUND(AVG(CASE WHEN decile = 10 THEN charges END), 2) AS avg_top_10pct_charges

FROM bucketed
GROUP BY region, smoker
ORDER BY region, smoker;


<details>
<summary><b> Solution (click to expand)</b></summary>

```sql
WITH bucketed AS (
  SELECT
    region, smoker, charges,
    NTILE(10) OVER (PARTITION BY region ORDER BY charges) AS decile
  FROM vw_insurance_flat
)
SELECT
  region,
  smoker,
  COUNT(*) AS n_people,
  SUM(CASE WHEN decile = 10 THEN 1 ELSE 0 END) AS n_top_10pct,
  ROUND(AVG(CASE WHEN decile = 10 THEN charges END), 2) AS avg_top_10pct_charges
FROM bucketed
GROUP BY region, smoker
ORDER BY region, smoker;
```

### Explanation (read-out friendly)

**What this query answers:** “Within each region (separately), what do the top 10% charges look like, split by smoker status?”

This uses two advanced ideas: **NTILE** (bucketing into deciles) and **conditional aggregation**.

- `WITH bucketed AS (...)`  
  Another CTE to keep the bucketing step separate from the summarisation step.

Inside the CTE:

- `NTILE(10) OVER (PARTITION BY region ORDER BY charges) AS decile`  
  - `NTILE(10)` assigns each row to one of **10 buckets** (1 to 10), aiming for equal-sized groups.
  - `PARTITION BY region` means we create deciles **within each region** (so regions are comparable internally).
  - `ORDER BY charges` sorts low → high, so `decile = 10` represents the **highest ~10%** charges in that region.

After the CTE (the summary):

- `COUNT(*) AS n_people`  
  Total number of people in each `(region, smoker)` group (all deciles combined).

- `SUM(CASE WHEN decile = 10 THEN 1 ELSE 0 END) AS n_top_10pct`  
  This is “conditional counting”:
  - `CASE WHEN ... THEN 1 ELSE 0 END` turns each row into a 1 (if top decile) or 0 (otherwise).
  - `SUM(...)` adds those 1s, giving the number of rows in the **top decile**.

- `AVG(CASE WHEN decile = 10 THEN charges END)`  
  Conditional average:
  - When `decile = 10`, we return `charges`.
  - Otherwise, the `CASE` returns `NULL`.
  - `AVG(...)` ignores `NULL`s, so it averages only top-decile charges.
  - `ROUND(..., 2)` rounds to 2 decimals for readability.

- `GROUP BY region, smoker`  
  Produces one output row per `(region, smoker)` group.

- `ORDER BY region, smoker`  
  Sorts the output for easy comparison.

</details>


# Theory Recap (15–25 min)

## Join cheat-sheet
- **INNER JOIN**: only matches
- **LEFT JOIN**: keep left rows; right becomes NULL when missing
- **RIGHT JOIN**: not supported in SQLite → emulate by swapping tables + LEFT JOIN
- **FULL OUTER JOIN**: not supported in SQLite → emulate with two LEFT JOINs + `UNION ALL`
- **CROSS JOIN**: all combinations (Cartesian product)
- **SELF JOIN**: table joined to itself

## Practical checks
- Confirm table grain
- Check row counts before/after joins
- Look for duplicates (accidental many-to-many)


## Baseline view (2 min)
This view matches the original CSV shape.

In [ ]:
%%sql
SELECT * FROM vw_insurance_flat
LIMIT 10;


# Basic Joins (25–40 min)

## 1) INNER JOIN (10 min)
We join person + dimensions + fact (charges) into one row per person.


In [ ]:
%%sql
SELECT
  p.person_id, p.age, s.sex, p.bmi, p.children, sm.smoker, rg.region, f.charges
FROM person p
JOIN insurance_fact f ON f.person_id = p.person_id
JOIN dim_sex s        ON s.sex_id = p.sex_id
JOIN dim_smoker sm    ON sm.smoker_id = p.smoker_id
JOIN dim_region rg    ON rg.region_id = p.region_id
LIMIT 10;


## 2) LEFT JOIN (5 min)
Keep all people, even if charges are missing (concept).  
In the raw dataset, everyone has charges, so it looks similar to INNER JOIN.


In [ ]:
%%sql
SELECT p.person_id, p.age, f.charges
FROM person p
LEFT JOIN insurance_fact f ON f.person_id = p.person_id
LIMIT 10;


# Intermediate Joins (40–55 min)

SQLite doesn’t support RIGHT/FULL OUTER joins directly. We show **emulation patterns** and use the demo views in the DB to make differences obvious.


## RIGHT JOIN emulation (5 min): swap + LEFT JOIN

In [ ]:
%%sql
SELECT f.person_id, p.age, f.charges
FROM insurance_fact f
LEFT JOIN person p ON p.person_id = f.person_id
LIMIT 10;


## FULL OUTER JOIN emulation (5 min): two LEFT JOINs + UNION ALL

In [ ]:
%%sql
SELECT p.person_id, f.person_id AS fact_person_id, p.age, f.charges
FROM person p
LEFT JOIN insurance_fact f ON f.person_id = p.person_id

UNION ALL

SELECT p.person_id, f.person_id AS fact_person_id, p.age, f.charges
FROM insurance_fact f
LEFT JOIN person p ON p.person_id = f.person_id
WHERE p.person_id IS NULL;


## Demo views (5 min): see NULLs and orphans
These views are already in `insurance.db` to make the join differences visible.


In [ ]:
%%sql
SELECT 'INNER' AS join_type, COUNT(*) AS rows FROM vw_inner_join_demo
UNION ALL
SELECT 'LEFT', COUNT(*) FROM vw_left_join_demo
UNION ALL
SELECT 'RIGHT (emulated)', COUNT(*) FROM vw_right_join_demo
UNION ALL
SELECT 'FULL OUTER (emulated)', COUNT(*) FROM vw_full_outer_join_demo;


In [ ]:
%%sql
-- LEFT JOIN demo: where charges are missing
SELECT * FROM vw_left_join_demo
WHERE charges IS NULL
LIMIT 20;


In [ ]:
%%sql
-- RIGHT JOIN demo: orphan facts (no matching person)
SELECT * FROM vw_right_join_demo
WHERE age IS NULL
ORDER BY person_id;


# Advanced Joins (55–70 min)

## CROSS JOIN (8 min)
We generate a region × smoker grid and LEFT JOIN stats onto it.


In [ ]:
%%sql
SELECT * FROM vw_cross_join_region_smoker_grid
ORDER BY region, smoker;


## SELF JOIN (7 min)
Find pairs of people in the same region with similar BMI.


In [ ]:
%%sql
SELECT * FROM vw_self_join_similar_bmi_pairs
ORDER BY bmi_diff ASC, region
LIMIT 50;


# Advanced Patterns (70–85 min)

Window functions are common in analytics:
- ranking within groups
- percentiles / deciles


In [ ]:
%%sql
-- Top 3 charges per region
WITH ranked AS (
  SELECT
    person_id, region, smoker, charges,
    ROW_NUMBER() OVER (PARTITION BY region ORDER BY charges DESC) AS rn
  FROM vw_insurance_flat
)
SELECT *
FROM ranked
WHERE rn <= 3
ORDER BY region, rn;


In [ ]:
%%sql
-- Deciles within region (inspect distribution)
SELECT region,
       NTILE(10) OVER (PARTITION BY region ORDER BY charges) AS decile,
       charges
FROM vw_insurance_flat
LIMIT 30;


# Wrap-up (85–90 min)

- Revisit the four assessment questions and show solutions (already hidden above).
- Ask: which join would you use in a real data pipeline and why?
- Homework: build BMI bands and compare charges by band × smoker × region.
